### pQTL tiering based on peptide level evidence

In [ ]:
import pandas as pd
import numpy as np 
import seaborn as sns
import matplotlib.pyplot as plt
from bioinfokit import analys, visuz
import os 
import pickle
import glob
from tqdm import tqdm
from src.config import intersection, lookup, diff

### Define directories

In [ ]:
# Define directories
Base='/home/projects/cpr_man/people/lilniu/projects/target/data/data_genotype/subset/Formatted/plink_qc4/gemma_peptide/'
Base1='/home/projects/cpr_man/people/lilniu/projects/target/data/data_genotype/subset/Formatted/plink_qc4/gemma/'
result_path = Base + 'results_final/'

### Read sig. pQTLs

In [ ]:
# Primary pQTLs
primary_pqtls = pd.read_csv(os.path.join(Base1, 'analysis/dataset/df_sig_primary.csv'))
pqtls=primary_pqtls['pqtl_id'].tolist()
# All pQTLs
df_all_pqtls = pd.read_csv(os.path.join(Base1, 'analysis/dataset/all_sig_pqtls.csv'))
all_pqtls=df_all_pqtls['pqtl_id'].unique().tolist()

### Read peptide info

In [ ]:
peptide_import=os.path.join(Base1, 'analysis/dataset/data_pep_export.pkl')
with open(peptide_import, 'rb') as handle:
    peptide_import = pickle.load(handle)
peptide_ids = peptide_import['peptide_ids']
peptide_ids['PEP.LeadPeptidePosition']=peptide_ids['PEP.PeptidePosition'].str.split(';').str[0]
peptide_ids=peptide_ids.rename({'PeptideID':'phenotype'}, axis=1)
peptide_ids['phenotype_pos']=peptide_ids['phenotype'].astype(str)+'_' + peptide_ids['PEP.LeadPeptidePosition'].astype(str)
peptide_ids = peptide_ids.drop_duplicates(subset=['PEP.StrippedSequence'])

In [ ]:
phenotype_to_proteinID = pd.read_csv(os.path.join(Base,'PeptideID.txt'), sep='\t')
print('Expecting {} assoc files in this experiment'.format(len(phenotype_to_proteinID)))
IDmapping_phenotype_to_proteinID = dict(zip(phenotype_to_proteinID['Phenotype ID'], phenotype_to_proteinID['Protein ID']))

### Read variants with coding consequences

In [ ]:
coding_variant = pd.read_csv(os.path.join(Base1, 'analysis/alphamap/coding_cons_allsig.csv'))

### Read peptide position data

In [ ]:
IDmapping_phenotype_to_pos = dict(zip(peptide_ids['phenotype'], peptide_ids['phenotype_pos']))

### Read assoc files

In [ ]:
# Read summary statistics after filtering for genome-wide significance (<5x10-8)
assoc_folder = 'assoc/'
os.chdir(os.path.join(result_path,assoc_folder))
assoc_files = glob.glob('*.txt')
assoc_phenotypes = [int(i.split('.')[0]) for i in assoc_files]
print('Numer of files: {}'.format(len(assoc_files)))

##### Check if any assoc files are missing

In [ ]:
missing_in_assoc = diff(phenotype_to_proteinID['Phenotype ID'].astype(int).tolist(), assoc_phenotypes)
print('Missing {} assoc files'.format(len(missing_in_assoc)))
print(missing_in_assoc)

### Read summary statistics

In [ ]:
# Read summary statistics 
sig_folder = 'sig'
os.chdir(os.path.join(result_path,sig_folder))
sig_files = glob.glob('*.txt')
sig_phenotypes = [int(i.split('_')[0]) for i in sig_files]
print('{} sig files'.format(len(sig_files)))

##### Check if any sig files with correponding assoc files are missing

In [ ]:
missing_in_sig = diff(assoc_phenotypes, sig_phenotypes)
print(missing_in_sig)

for i in missing_in_sig:
    print(i)

In [ ]:
files = []
for i in tqdm(sig_files):
    df = pd.read_csv(os.path.join(result_path,sig_folder,i), sep='\t').drop(['Unnamed: 0'], axis=1)
    if not df.empty:
        df = df
    else:
        phenotypeID=int(i.split('_')[0])
        phenotype=IDmapping_phenotype_to_proteinID[phenotypeID]
        df.loc[0, 'phenotype']=phenotype
    files.append(df)
    
# Check if any file is missing
print('Numer of files: {}'.format(len(sig_files)))

In [ ]:
# Combine all significant hits
df_files = pd.concat(files)
cols_tokeep = ['phenotype', 'Protein ID', 'Gene name', 'PEP.StrippedSequence',
               'PEP.UsedForProteinGroupQuantity', 'PEP.IsProteinGroupSpecific', ]

df_files = df_files.merge(peptide_ids[cols_tokeep], on='phenotype', how='left')
df_files['pqtl_id'] = df_files['rs'].astype(str) +'_' + df_files['Protein ID'].astype(str) + '_' + df_files['Gene name'].astype(str)
df_files['REF'] = df_files['rs'].str.split('_').str[2]
df_files['phenotype_pos']=df_files['phenotype'].map(IDmapping_phenotype_to_pos)
df_files = df_files[df_files['PEP.IsProteinGroupSpecific']=='True']

# Check if any reference and alternative SNPs are flipped

check_flip = df_files.dropna()
flipped = check_flip[check_flip['REF']!= check_flip['allele0']]
print('Reference and alternative SNPs were flipped in the following rows: {}'.format(flipped))

#### Get peptides available for test

In [ ]:
pep_avail1 = df_files[['Protein ID', 'phenotype_pos', 'PEP.UsedForProteinGroupQuantity', 'PEP.IsProteinGroupSpecific']].drop_duplicates()
pep_avail = pep_avail1.groupby('Protein ID', as_index=False)['phenotype_pos'].agg(list).rename({'phenotype_pos': 'peptides available for test'}, axis=1)
pep_avail['nr.peptides.avail'] = pep_avail['peptides available for test'].apply(len)
pep_avail.to_csv(os.path.join(Base1, 'analysis/annotation/pep_avail.csv'))

In [ ]:
df_sig=df_files.dropna(subset='beta')

In [ ]:
df_sig=df_sig[df_sig['p_wald']<0.05/primary_pqtls.shape[0]]

#### Any proteins whose peptides are sig. but not at protein level?

In [ ]:
nr_peptides_gwas = peptide_ids[(peptide_ids['gwas']==True) &(peptide_ids['PEP.IsProteinGroupSpecific']=='True')].shape[0]
pep_sig_proteinIDs = df_sig[(df_sig['p_wald']<5e-8/nr_peptides_gwas)&(~df_sig['p_wald'].isna())]['Protein ID'].unique().tolist()
pro_sig_proteinIDs = primary_pqtls['Protein ID'].unique().tolist()
new_proteins = [i for i in pep_sig_proteinIDs if i not in pro_sig_proteinIDs]
old_proteins = [i for i in pep_sig_proteinIDs if i not in new_proteins]

dict_proteins_found_ornot = {x:'old' for x in df_all_pqtls['Protein ID'].unique()}
dict_proteins_found_ornot.update({x:'new' for x in new_proteins})

In [ ]:
len(new_proteins)

In [ ]:
LIMIT_TO_FOUND_PQTLS = False
if LIMIT_TO_FOUND_PQTLS:
    df_sig_check = lookup(df_sig, 'pqtl_id', all_pqtls)
else:
    df_sig_check=df_sig

In [ ]:
pep_sig = df_sig_check[['Protein ID', 'phenotype_pos', 'pqtl_id']].drop_duplicates()
pep_sig = pep_sig.groupby('pqtl_id', as_index=False)['phenotype_pos'].agg(list).rename({'phenotype_pos': 'peptides.sig'}, axis=1)
pep_sig['nr.peptides.sig'] = pep_sig['peptides.sig'].apply(len)

In [ ]:
pep_sig_info = df_sig_check.groupby('pqtl_id', as_index=False)['PEP.IsProteinGroupSpecific'].agg(list).rename({'PEP.IsProteinGroupSpecific': 'sig.PG.spec.'}, axis=1)

In [ ]:
pep_sig=pep_sig.merge(pep_sig_info, on='pqtl_id', how='left')

In [ ]:
pep_sig['nr.peptides.sig.spec'] = pep_sig['sig.PG.spec.'].apply(lambda x: x.count('True'))

In [ ]:
# Check direction
def check_direction(series):
    all_positive = all(x > 0 for x in series)
    all_negative = all(x < 0 for x in series)
    positive_count = sum(x > 0 for x in series)
    negative_count = sum(x < 0 for x in series)
    # If all values are either all positive or all negative, return True
    return all_positive or all_negative, positive_count, negative_count

# Apply the custom function to each group
direction_check = df_sig_check.groupby('pqtl_id')['beta'].apply(check_direction).reset_index(name='Result')
direction_check[['all.same.direction', 'pos.count', 'neg.count']] = pd.DataFrame(direction_check['Result'].tolist(), index=direction_check.index)

In [ ]:
pep_sig['Protein ID']=pep_sig['pqtl_id'].str.split('_').str[4]
pep_sig = pep_sig.merge(direction_check, on='pqtl_id', how='left')
pep_sig = pep_sig.merge(pep_avail, on='Protein ID', how='left')
pep_sig['%.sig'] = pep_sig['nr.peptides.sig.spec']/pep_sig['nr.peptides.avail']
get_last_element = lambda x: [item.split('_')[-1] for item in x if not pd.isna(item)] if isinstance(x, list) else []
pep_sig['peptides.sig.pos']=pep_sig['peptides.sig'].apply(get_last_element)

In [ ]:
pep_sig['protein_sig']=pep_sig['Protein ID'].map(dict_proteins_found_ornot).fillna('not.sig')

In [ ]:
df_evidence = df_all_pqtls.merge(pep_sig.drop('Protein ID', axis=1), how='left', on='pqtl_id')
df_evidence = df_evidence.assign(pqtl_direction=np.where(df_evidence['beta']>0, 'pos', 'neg'))
df_evidence = df_evidence.assign(primary_or_not=np.where(df_evidence['pqtl_id'].isin(pqtls), True, False))

In [ ]:
#Add coding variant annotation
df_evidence = df_evidence.merge(coding_variant, how='left', left_on='rs', right_on='#Uploaded_variation')

#### Number of pqtls with concordant evidence at peptide level with at least one peptide

In [ ]:
#tier1 pQTL condition
cond1=(df_evidence['pos.count']>1) & (df_evidence['pqtl_direction']=='pos') 
cond2=(df_evidence['neg.count']>1) & (df_evidence['pqtl_direction']=='neg')
tier1_cond = (cond1 | cond2)

#tier2 pQTL condition
cond3=(df_evidence['nr.peptides.sig']==1) & (df_evidence['Gene name']!=df_evidence['SYMBOL'])
cond4=(~df_evidence['SYMBOL'].isna()) & (df_evidence['nr.peptides.avail']<4)
cond3_1 = (df_evidence['pos.count']==1) & (df_evidence['pqtl_direction']=='pos')
cond3_2 = (df_evidence['neg.count']==1) & (df_evidence['pqtl_direction']=='neg')

tier2_cond = (cond3 & cond4) & (cond3_1 | cond3_2)

In [ ]:
tier1=df_evidence[tier1_cond]
tier2=df_evidence[tier2_cond]

In [ ]:
detective_pipes = ['pqtl_id', 'all.same.direction', 'pos.count', 'neg.count', 
                   'nr.peptides.avail', 'nr.peptides.sig','peptides available for test','%.sig',
              'primary_or_not', 'peptides.sig', 'Protein_position', 'Amino_acids','SYMBOL', 'nr.peptides.sig.spec']

#### Number of pqtls identified with single peptides which need further inspection re artefactual pqtls

In [ ]:
tier3_totest_cond = (df_evidence['nr.peptides.sig']==1) & (df_evidence['Gene name']==df_evidence['SYMBOL'])

In [ ]:
tier3_totest = df_evidence[tier3_totest_cond]

In [ ]:
false_ids = ['P40189_IL6ST', 'P35916_FLT4', 'P61626_LYZ', 'Q99650_OSMR', ]

In [ ]:
tier3 = tier3_totest[~tier3_totest['phenotype'].isin(false_ids)]

In [ ]:
tier4_cond = (df_evidence['SYMBOL'].isna()) & (df_evidence['nr.peptides.sig']==1) & (df_evidence['nr.peptides.avail']<4) & (~df_evidence['phenotype'].isin(false_ids))

In [ ]:
tier4_totest = df_evidence[tier4_cond]
tier4 = tier4_totest[tier4_totest['nr.peptides.sig.spec']>0]

In [ ]:
tier1_missense = tier1[~tier1['SYMBOL'].isna()][detective_pipes]

In [ ]:
pep_sig['Gene name']=pep_sig['pqtl_id'].str.split('_').str[-1]
pep_sig['chr']=pep_sig['pqtl_id'].str.split('_').str[0].astype(int)

#### Eliminate artefactual pQTLs

In [ ]:
tier1['tier']='tier1'
tier2['tier']='tier2'
tier3['tier']='tier3'
tier4['tier']='tier4'

In [ ]:
df_final = pd.concat([tier1, tier2, tier3, tier4]).drop_duplicates(subset=['pqtl_id'])

In [ ]:
df_final_primary=lookup(df_final, 'primary_or_not', True)

In [ ]:
df_final_primary[df_final_primary['tier']=='tier1']['%.sig'].median()

In [ ]:
tier1_final = df_final_primary[df_final_primary['tier']=='tier1']

In [ ]:
df_final_primary[df_final_primary['tier']=='tier1']['%.sig'].hist(bins=20)
plt.xlabel('Nr. observed sig. peptides/Nr. peptides availble for testing')
plt.ylabel('Nr. pQTLs')
plt.title('Tier1 peptide evidence')

In [ ]:
df_final_primary.to_csv(os.path.join(Base1, 'analysis/alphamap/astral/final_set.csv'), index=False)

In [ ]:
df_final.to_csv(os.path.join(Base1, 'analysis/results/df_sig_all_final.csv'), index=False)

In [ ]:
lookup(tier1, 'primary_or_not', True)['all.same.direction'].value_counts(1)